# E2-prompt: does a global (prompted) readout restore convergence with scale across 20 LLMs?

**Motivation.** In E6 (Qwen3, 0.6B → 8B), reading the decoder with a PromptEOL prompt raised calibrated CKA with DINOv2 from 0.23 to 0.37 under the Aristotelian pipeline. Mean pooling (the Huh / Aristotelian readout) stayed flat. This notebook asks whether that holds across the 20 LLMs of the E2 audit. It reuses the E2 pipeline unchanged: WIT-1024, the 17 ViTs (their E2 features on Drive), all layer pairs, a permutation null on the max, 200 permutations, exact bf16, CPU offload for the largest models.

**Readouts.** All are the last token of a template, at every layer:
- `eol` (main): `This sentence : "<caption>" means in one word:"`
- `eol_alt`: a second summarising prompt, `This image caption : "<caption>" can be summarized in one word:"`
- `neutral` (control): the same frame without a summary request, `This sentence : "<caption>" was written down and then`
- `repeat` (control): the caption twice, `<caption>\n<caption>`, so the last token has seen the whole caption, with no instruction

The mean-pooled baseline is the E2 result itself (`prh_audit/full/results/alignment_all.csv`). Layer 0 of a fixed-final-token readout is the same vector for every caption, so zero-variance layers are dropped from the layer search, as in E6 v2.

**Pre-registered predictions** (written before any E2-prompt result; primary metric = calibrated CKA, Huh preprocessing, layer search):
- **R1 (scale):** the slope over log2(parameters), with family and ViT fixed effects, is larger for `eol` than for `mean`. Two-way bootstrap (LLMs × ViTs) CI of the difference excludes 0.
- **R2 (capability):** the Pearson correlation of LLM means with capability (−bits per byte) is larger for `eol` than for `mean`, two-way CI excluding 0.
- **R3 (it is the summary request):** `eol` − `neutral` > 0 (mean over all pairs, CI excludes 0), and `eol_alt` ≈ `eol` (|Δ| < 0.02).
- **R4 (descriptive):** inside the families with a wide size range (LLaMA 7→65B, Llama-3 8→70B, Mistral 7B→Mixtral), `eol` rises where `mean` was flat.

**How to run.**
1. `SMOKE_TEST = True` (default): BLOOMZ-560m and OLMo-1B × 2 ViTs, all readouts, 50 permutations, ~10 min. It ends with a CHECKER table and `e2p_smoke_results.zip`.
2. Full run: set `SMOKE_TEST = False`. Small and mid-size models first (`RUN_EXCEPT` the two biggest), then LLaMA-65B and Llama-3-70B one per session with `RUN_ONLY`, as in v4. The last session writes `e2p_results.zip`. Accept the Gemma and Mistral licences with the same HF token as before.
3. To save time on the 65B/70B sessions, `CONTROLS_MAX_GB` limits the control readouts to models up to that size (default 40 GB of weights). The main `eol` readout runs on all 20.

In [ ]:
# ============================== CONFIG ==============================
import os
SMOKE_TEST = True
READOUTS = ("eol", "eol_alt", "neutral", "repeat")
CONTROLS_MAX_GB = 40.0       # eol on every model; eol_alt / neutral / repeat only on models with <= this many GB of weights
N_PERMS = 200
ALPHA = 0.05
TOPK = 10
VARIANTS_P = ("huh", "noclip")
B_BOOT = 2000
SEED = 0
RUN_ONLY = None              # e.g. "huggyllama/llama-65b": extract only this model in this session
RUN_EXCEPT = None            # e.g. ["huggyllama/llama-65b", "NousResearch/Meta-Llama-3-70B"]
LLMS_ALL = ["bigscience/bloomz-560m", "bigscience/bloomz-1b1", "bigscience/bloomz-1b7", "bigscience/bloomz-3b",
            "bigscience/bloomz-7b1", "openlm-research/open_llama_3b", "openlm-research/open_llama_7b",
            "openlm-research/open_llama_13b", "huggyllama/llama-7b", "huggyllama/llama-13b",
            "huggyllama/llama-30b", "huggyllama/llama-65b", "allenai/OLMo-1B-hf", "allenai/OLMo-7B-hf",
            "google/gemma-2b", "google/gemma-7b", "mistralai/Mistral-7B-v0.1", "mistralai/Mixtral-8x7B-v0.1",
            "NousResearch/Meta-Llama-3-8B", "NousResearch/Meta-Llama-3-70B"]
VISION_ALL = ["vit_tiny_patch16_224.augreg_in21k", "vit_small_patch16_224.augreg_in21k",
              "vit_base_patch16_224.augreg_in21k", "vit_large_patch16_224.augreg_in21k",
              "vit_base_patch16_224.mae", "vit_large_patch16_224.mae", "vit_huge_patch14_224.mae",
              "vit_small_patch14_dinov2.lvd142m", "vit_base_patch14_dinov2.lvd142m",
              "vit_large_patch14_dinov2.lvd142m", "vit_giant_patch14_dinov2.lvd142m",
              "vit_base_patch16_clip_224.laion2b", "vit_large_patch14_clip_224.laion2b",
              "vit_huge_patch14_clip_224.laion2b", "vit_base_patch16_clip_224.laion2b_ft_in12k",
              "vit_large_patch14_clip_224.laion2b_ft_in12k", "vit_huge_patch14_clip_224.laion2b_ft_in12k"]
if SMOKE_TEST:
    LLMS = ["bigscience/bloomz-560m", "allenai/OLMo-1B-hf"]
    VISION = ["vit_tiny_patch16_224.augreg_in21k", "vit_small_patch14_dinov2.lvd142m"]
    N_PERMS, B_BOOT = 50, 200
else:
    LLMS, VISION = LLMS_ALL, VISION_ALL

def llm_family(m):
    m = m.lower()
    for key, fam in (("bloomz", "BLOOMZ"), ("open_llama", "OpenLLaMA"), ("huggyllama", "LLaMA"), ("olmo", "OLMo"),
                     ("gemma", "Gemma"), ("mixtral", "Mistral"), ("mistral", "Mistral"), ("llama-3", "Llama-3")):
        if key in m: return fam
    return "other"

In [ ]:
# ============================== SETUP ==============================
import sys, subprocess, json, time, glob, shutil, math
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "timm", "accelerate", "psutil",
                    "huggingface_hub", "datasets", "sentencepiece", "protobuf"], check=False)
    from google.colab import drive; drive.mount("/content/drive")
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
ROOT = os.environ.get("PRH_ROOT") or "/content/drive/MyDrive/prh_audit"
E2_FEAT = os.path.join(ROOT, "full", "features")                  # E2 vision features (CLS per block), reused
E2_RES = os.path.join(ROOT, "full", "results")                    # E2 mean-pooled results + bpb, reused
RUN = os.path.join(ROOT, "e2prompt_smoke" if SMOKE_TEST else "e2prompt")
FEAT_DIR, RES_DIR, FIG_DIR = (os.path.join(RUN, d) for d in ("features", "results", "figs"))
for d in (FEAT_DIR, RES_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
import torch, numpy as np, pandas as pd
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LOG = os.path.join(RUN, "log.txt")
def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a); print(s); open(LOG, "a").write(s + "\n")
import transformers, psutil
log("device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu", "| transformers", transformers.__version__,
    "| free disk GB", round(shutil.disk_usage("/").free / 1e9), "| run dir:", RUN)
assert all(os.path.exists(os.path.join(E2_FEAT, v.replace("/", "_") + ".pt")) for v in VISION), "E2 vision features not found on Drive"
assert os.path.exists(os.path.join(E2_RES, "alignment_all.csv")), "E2 results (alignment_all.csv) not found on Drive"

In [ ]:
# ============================== ENGINE (identical to E2, v3) ==============================
# ===================== Alignment engine (all layer pairs at once, GPU) =====================
# Every metric is computed for ALL (layer_x, layer_y) pairs with one matmul on flattened
# n x n matrices. Permutation nulls permute one side's matrices with the SAME permutation
# for every layer, so the max-over-layer-pairs aggregate is calibrated as in the Aristotelian paper.
import math, numpy as np, torch

VARIANTS = ("huh", "noclip", "std")
# huh    : clamp |x| at the per-sample 95th percentile (averaged), then L2-normalise each sample
#          (exactly Huh et al.'s prepare_features + compute_score(normalize=True))
# noclip : L2-normalise only
# std    : per-layer, per-dimension z-score over samples, then L2-normalise

def huh_clip(feats, q=0.95):
    """Huh et al. remove_outliers(exact=False): per-sample q-quantile of |x| over all layers & dims
    (linear interpolation, as torch.quantile), averaged over samples, then clamp. Uses kthvalue so it
    also works on tensors larger than torch.quantile's size limit."""
    a = feats.abs().flatten(start_dim=1); m = a.shape[1]
    pos = q * (m - 1); lo = int(math.floor(pos)); hi = int(math.ceil(pos)); w = pos - lo
    qs = []
    for i in range(a.shape[0]):
        v_lo = a[i].kthvalue(lo + 1).values
        v_hi = a[i].kthvalue(hi + 1).values if hi != lo else v_lo
        qs.append(v_lo + (v_hi - v_lo) * w)
    t = torch.stack(qs).mean()
    return feats.clamp(-t, t)

def preprocess(feats, variant):
    """feats: N x L x D float32 tensor -> list of L tensors (N x D), ready for metrics."""
    if variant == "huh":
        f = huh_clip(feats)
    elif variant == "noclip":
        f = feats
    elif variant == "std":
        f = (feats - feats.mean(0, keepdim=True)) / (feats.std(0, keepdim=True) + 1e-6)
    else:
        raise ValueError(variant)
    return [torch.nn.functional.normalize(f[:, l, :], dim=-1) for l in range(f.shape[1])]

def _center(K):
    m = K.mean(0)
    return K - m[None, :] - m[:, None] + m.mean()

class SideStats:
    """Per-model, per-variant matrices needed for every metric, flattened over layers."""
    def __init__(self, layers, k=10, half_idx=None):
        n = layers[0].shape[0]; self.n = n; self.L = len(layers); self.k = k
        Kc, Kt, adj, reff = [], [], [], []
        Kt_a, Kt_b, Kc_a, Kc_b = [], [], [], []
        ia, ib = half_idx if half_idx is not None else (None, None)
        for X in layers:
            K = X @ X.T
            c = _center(K); Kc.append(c.flatten())
            Kd = X.double() @ X.double().T                                     # fp64: unbiased HSIC cancels large terms
            t = Kd.clone(); t.fill_diagonal_(0); Kt.append(t.flatten())
            reff.append((torch.trace(c) ** 2 / (c * c).sum()).item())
            S = K.clone(); S.fill_diagonal_(-1e8)
            idx = S.topk(k, dim=1).indices
            A = torch.zeros(n, n, device=X.device); A.scatter_(1, idx, 1.0); adj.append(A.flatten())
            if ia is not None:
                for sub, Kt_s, Kc_s in ((ia, Kt_a, Kc_a), (ib, Kt_b, Kc_b)):
                    Ks = K[sub][:, sub]; Kc_s.append(_center(Ks).flatten())
                    ts = Kd[sub][:, sub].clone(); ts.fill_diagonal_(0); Kt_s.append(ts.flatten())
        self.Kc = torch.stack(Kc); self.Kc_norm = self.Kc.norm(dim=1)
        self.Kt = torch.stack(Kt); self.adj = torch.stack(adj)
        self.reff = np.array(reff)
        self.hu = self._hsic_parts(self.Kt, n)
        if ia is not None:
            self.Kc_a = torch.stack(Kc_a); self.Kc_b = torch.stack(Kc_b)
            self.Kc_a_norm = self.Kc_a.norm(dim=1); self.Kc_b_norm = self.Kc_b.norm(dim=1)
            self.hu_a = self._hsic_parts(torch.stack(Kt_a), len(ia))
            self.hu_b = self._hsic_parts(torch.stack(Kt_b), len(ib))

    @staticmethod
    def _hsic_parts(Kt, n):
        M = Kt.view(Kt.shape[0], n, n)
        return dict(flat=Kt, tot=Kt.sum(1), rows=M.sum(2), n=n)

def _hsic_u_pairs(A, B):
    """Unbiased HSIC (Song et al. 2012) for all layer pairs. A, B: dicts from _hsic_parts."""
    n = A["n"]
    t1 = A["flat"] @ B["flat"].T
    t2 = A["tot"][:, None] * B["tot"][None, :] / ((n - 1) * (n - 2))
    t3 = 2.0 * (A["rows"] @ B["rows"].T) / (n - 2)
    return (t1 + t2 - t3) / (n * (n - 3))

def _hsic_u_self(A):
    n = A["n"]
    return ((A["flat"] ** 2).sum(1) + A["tot"] ** 2 / ((n - 1) * (n - 2))
            - 2.0 * (A["rows"] ** 2).sum(1) / (n - 2)) / (n * (n - 3))

def unbiased_cka_pairs(A, B, rel_tol=1e-6):
    """fp64. Layers whose self-HSIC is ~0 or negative (near-constant representations, e.g. a ViT's
    first-block CLS token) are undefined for CKA and returned as NaN instead of dividing by ~0."""
    sa, sb = _hsic_u_self(A), _hsic_u_self(B)
    va = sa > rel_tol * sa.max().clamp_min(1e-300); vb = sb > rel_tol * sb.max().clamp_min(1e-300)
    u = _hsic_u_pairs(A, B) / torch.sqrt(sa.clamp_min(1e-300)[:, None] * sb.clamp_min(1e-300)[None, :])
    u[~va, :] = float("nan"); u[:, ~vb] = float("nan")
    return u.float()

def _nanargmax2(M):
    i = int(torch.nan_to_num(M, nan=-float("inf")).argmax()); return divmod(i, M.shape[1])

def linear_cka_pairs(x, y):
    return (x.Kc @ y.Kc.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])

def mknn_pairs(x, y):
    return (x.adj @ y.adj.T) / (x.n * x.k)

def _perm_flat_index(p, n):
    return (p[:, None] * n + p[None, :]).flatten()

def calibrate(obs, null, alpha=0.05, smax=1.0):
    K = len(null)
    allv = np.sort(np.append(null, obs))
    tau = allv[int(math.ceil((1 - alpha) * (K + 1))) - 1]
    cal = max((obs - tau) / (smax - tau), 0.0) if smax > tau else 0.0
    p = (1 + np.sum(null >= obs)) / (K + 1)
    return cal, tau, p

def align_pair(x, y, perms, alpha=0.05, perms_half=None):
    """x: language SideStats, y: vision SideStats. Returns dict of scalar results + matrices."""
    n = x.n
    cka = linear_cka_pairs(x, y); ucka = unbiased_cka_pairs(x.hu, y.hu); mk = mknn_pairs(x, y)
    i_c, j_c = divmod(int(cka.argmax()), cka.shape[1])
    i_u, j_u = _nanargmax2(ucka)
    i_m, j_m = divmod(int(mk.argmax()), mk.shape[1])
    null_cka_max, null_mk_max, null_cka_fixed, null_arg = [], [], [], []
    for p in perms:
        fi = _perm_flat_index(p, n)
        Kc_p = y.Kc.index_select(1, fi); A_p = y.adj.index_select(1, fi)
        s = (x.Kc @ Kc_p.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])
        a_ = int(s.argmax()); null_arg.append(a_)
        null_cka_max.append(s.view(-1)[a_].item()); null_cka_fixed.append(s[i_c, j_c].item())
        null_mk_max.append(((x.adj @ A_p.T) / (n * x.k)).max().item())
    cka_max = cka.max().item(); mk_max = mk.max().item()
    cal_agg, tau_agg, p_agg = calibrate(cka_max, np.array(null_cka_max), alpha)
    cal_fix, tau_fix, _ = calibrate(cka_max, np.array(null_cka_fixed), alpha)
    mk_cal, mk_tau, mk_p = calibrate(mk_max, np.array(null_mk_max), alpha)
    out = dict(cka_max=cka_max, cka_cal=cal_agg, cka_tau=tau_agg, cka_p=p_agg,
               cka_cal_fixedpair=cal_fix, cka_tau_fixedpair=tau_fix,
               cka_null_mean=float(np.mean(null_cka_max)),
               ucka_max=float(np.nanmax(ucka.cpu().numpy())), mknn_max=mk_max, mknn_cal=mk_cal, mknn_tau=mk_tau, mknn_p=mk_p,
               best_cka=(i_c, j_c), best_ucka=(i_u, j_u), best_mknn=(i_m, j_m),
               reff_x=float(x.reff[i_c]), reff_y=float(y.reff[j_c]))
    out["pred_shrink"] = 1.0 / math.sqrt((1 + out["reff_x"] / n) * (1 + out["reff_y"] / n))
    # which layer pair sets the aggregate null threshold most often, and how low-rank is it?
    mode = max(set(null_arg), key=null_arg.count); im, jm = divmod(mode, cka.shape[1])
    out.update(nullmax_pair=(im, jm), nullmax_pair_freq=null_arg.count(mode) / len(null_arg),
               nullmax_reff_x=float(x.reff[im]), nullmax_reff_y=float(y.reff[jm]))
    if hasattr(x, "hu_a") and hasattr(y, "hu_a"):
        ua = unbiased_cka_pairs(x.hu_a, y.hu_a)
        ia, ja = _nanargmax2(ua)
        ub = unbiased_cka_pairs(x.hu_b, y.hu_b)
        out["ucka_heldout"] = ub[ia, ja].item(); out["best_heldout"] = (ia, ja)
        # held-out CALIBRATED CKA: pick the layer pair on half A by UNBIASED CKA (raw CKA is inflated by the
        # chance offset and would favour high-rank layers), then calibrate that single pair's raw CKA on half B
        ic, jc = ia, ja
        xb, yb = x.Kc_b[ic], y.Kc_b[jc]; nb = int(round(math.sqrt(xb.numel())))
        obs = (xb @ yb / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
        nulls = [(xb @ yb.index_select(0, _perm_flat_index(p, nb)) / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
                 for p in (perms_half or [])]
        if nulls:
            c_, t_, p_ = calibrate(obs, np.array(nulls), alpha)
            out.update(cka_heldout_raw=obs, cka_heldout_cal=c_, cka_heldout_tau=t_, best_heldout_cka=(ic, jc))
    mats = dict(cka=cka.cpu().numpy(), ucka=ucka.cpu().numpy(), mknn=mk.cpu().numpy())
    return out, mats

def layer_diagnostics(feats):
    """Outlier diagnostics on raw (un-normalised) features. feats: N x L x D."""
    rows = []
    for l in range(feats.shape[1]):
        f = feats[:, l, :]
        v = f.var(0); vs = torch.sort(v, descending=True).values; tot = v.sum()
        absf = f.abs()
        fc = f - f.mean(0, keepdim=True)
        G = fc @ fc.T
        rows.append(dict(layer=l, top1_var_share=(vs[0] / tot).item(), top10_var_share=(vs[:10].sum() / tot).item(),
                         max_over_median_abs=(absf.max() / absf.median().clamp_min(1e-12)).item(),
                         reff_raw=(torch.trace(G) ** 2 / (G * G).sum()).item()))
    return rows

In [ ]:
# ============================== EXTRACTION CODE (E2 v4 loader + prompted readouts) ==============================
# ===================== Feature extraction (Huh et al. protocol), exact bf16 =====================
# Language: every hidden state (embeddings + each block), mean-pooled over non-pad tokens,
#           left padding to the longest caption in the dataset, bf16 weights. NO quantization:
#           models larger than the GPU run with accelerate CPU/disk offload, which streams the bf16
#           weights to the GPU layer by layer, so the arithmetic is identical to an all-GPU run.
# Vision:   CLS token after every transformer block (= Huh's "blocks.{i}.add_1" node), timm transforms.
import gc, glob, math, os, shutil, time
import torch, transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

def _dtype_kw(dtype):
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}

def load_tokenizer(name, path=None):
    path = path or name
    try:
        tok = AutoTokenizer.from_pretrained(path)
    except Exception as e:                      # e.g. OpenLLaMA fast-tokenizer conversion issues
        print("  fast tokenizer failed, using slow:", repr(e)[:120])
        tok = AutoTokenizer.from_pretrained(path, use_fast=False)
    if "huggyllama" in name:
        tok.pad_token = tok.unk_token           # Huh et al. set "[PAD]" (not in vocab); padded positions are masked either way
    elif tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"
    return tok

def weights_gb(name):
    """Exact checkpoint size from the Hub metadata (safetensors if present, else .bin)."""
    from huggingface_hub import HfApi
    info = HfApi().model_info(name, files_metadata=True)
    files = [(s.rfilename, s.size or 0) for s in info.siblings]
    st = [sz for f, sz in files if f.endswith(".safetensors")]
    bn = [sz for f, sz in files if f.endswith(".bin") and "training_args" not in f]
    return (sum(st) if st else sum(bn)) / 1e9, ("safetensors" if st else "bin")

def plan_placement(name, gpu_headroom_gb=14.0, cpu_headroom_gb=16.0, disk_headroom_gb=8.0):
    """Decide GPU / GPU+CPU / GPU+CPU+disk placement for exact bf16 loading, or a reason to skip.
    Memory budgets are computed in GiB (2**30 bytes), the unit accelerate's max_memory uses; disk in GB.
    14 GiB GPU headroom = 7 GiB activations (extract_llm sizes the batch to fit) + CUDA contexts, the layer
    being streamed in from CPU during offload, and allocator fragmentation. Offload is bit-exact
    (verified on the GPU by the smoke test), so erring towards offload only costs time."""
    import psutil
    GiB = 2 ** 30
    w, fmt = weights_gb(name)                                   # GB (1e9), from the Hub metadata
    w_gib = w * 1e9 / GiB
    gpu = (torch.cuda.get_device_properties(0).total_memory / GiB) if torch.cuda.is_available() else 0.0
    cpu = psutil.virtual_memory().available / GiB
    disk = shutil.disk_usage("/").free / 1e9
    plan = dict(weights_gb=round(w, 1), weights_gib=round(w_gib, 1), format=fmt, gpu_gib=round(gpu, 1),
                cpu_free_gib=round(cpu, 1), disk_free_gb=round(disk, 1))
    if disk < w + disk_headroom_gb:
        return dict(plan, mode="skip", reason=f"download needs {w + disk_headroom_gb:.0f} GB disk, {disk:.0f} GB free")
    gpu_budget, cpu_budget = max(gpu - gpu_headroom_gb, 0), max(cpu - cpu_headroom_gb, 0)
    if w_gib <= gpu_budget:
        return dict(plan, mode="gpu", max_memory=None)
    mm = {0: f"{int(gpu_budget)}GiB", "cpu": f"{int(cpu_budget)}GiB"}
    if w_gib <= int(gpu_budget) + int(cpu_budget):
        return dict(plan, mode="gpu+cpu", max_memory=mm)
    spill_gb = (w_gib - int(gpu_budget) - int(cpu_budget)) * GiB / 1e9
    if disk - w - disk_headroom_gb > spill_gb * 1.1:
        return dict(plan, mode="gpu+cpu+disk", max_memory=mm)
    return dict(plan, mode="skip", reason=f"needs {spill_gb:.0f} GB disk offload beyond the download; not enough disk")

MODEL_DIR = "/content/hf_models"

def local_model_dir(name):
    return os.path.join(MODEL_DIR, name.replace("/", "--"))

def download_llm(name):
    """Download exactly one copy of the weights (the format weights_gb counted) into a dedicated folder.
    Skips duplicate formats (Meta's original/*.pth, .msgpack, .h5), which would double the disk needed."""
    from huggingface_hub import snapshot_download
    _, fmt = weights_gb(name)
    weights = ["*.safetensors"] if fmt == "safetensors" else ["*.bin"]
    return snapshot_download(name, local_dir=local_model_dir(name),
                             allow_patterns=["*.json", "*.model", "*.tiktoken", "*.txt"] + weights,
                             ignore_patterns=["original/*", "*.pth", "*.pt", "*.msgpack", "*.h5", "*.onnx",
                                              "training_args.bin"])

def load_llm(path, plan, offload_dir):
    kw = dict(device_map="auto", **_dtype_kw(torch.bfloat16))
    if plan["mode"] != "gpu":
        # JSON turns the GPU key 0 into "0"; accelerate needs the integer device index
        kw["max_memory"] = {(int(k) if str(k).isdigit() else k): v for k, v in plan["max_memory"].items()}
    if plan["mode"] == "gpu+cpu+disk":
        os.makedirs(offload_dir, exist_ok=True); kw["offload_folder"] = offload_dir
    return AutoModelForCausalLM.from_pretrained(path, **kw).eval()

@torch.no_grad()
def extract_llm(name, texts, plan, offload_dir="/content/offload", batch_size=32, act_budget_gb=7.0):
    path = download_llm(name)                   # one explicit folder per model, deleted by the notebook afterwards
    tok = load_tokenizer(name, path)
    model = load_llm(path, plan, offload_dir)
    n_params = sum(p.numel() for p in model.parameters())
    tokens = tok(texts, padding="longest", return_tensors="pt")
    T = tokens["input_ids"].shape[1]; V = model.get_output_embeddings().weight.shape[0]
    # bf16 logits + their fp32 copy for the loss: 6 bytes per (token, vocab) entry; keep under the budget
    # Batch size from the activation memory of THIS model, so it fits in the GPU headroom left by the weights.
    # Per token: hidden states of every layer (kept until the forward ends because output_hidden_states=True),
    # logits in bf16 + their fp32 copy for the loss, MLP intermediates, residual/attention scratch; per sequence:
    # attention scores (fp32 upper bound). The batch size never changes the per-caption math (no batch statistics).
    c = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
    L, D = c.num_hidden_layers, c.hidden_size
    I = getattr(c, "intermediate_size", None) or 4 * D
    H = getattr(c, "num_attention_heads", 1)
    per_seq = T * ((L + 1) * D * 2 + V * 6 + 4 * I * 2 + 8 * D * 2) + H * T * T * 4
    bs = max(1, min(batch_size, int(act_budget_gb * 1e9 // per_seq)))
    print(f"  activation estimate {per_seq / 1e6:.0f} MB per caption -> batch size {bs}", flush=True)
    dev = "cuda:0" if torch.cuda.is_available() else "cpu"   # accelerate hooks move inputs to each module's device
    feats, losses, bpbs = [], [], []
    crit = torch.nn.CrossEntropyLoss(reduction="none")
    for i in range(0, len(texts), bs):
        ids = tokens["input_ids"][i:i + bs].to(dev).long()
        am = tokens["attention_mask"][i:i + bs].to(dev).long()
        out = model(input_ids=ids, attention_mask=am, output_hidden_states=True)
        m = am[:, :-1].float()
        logits = out.logits[:, :-1].float().to(m.device)
        tok_loss = m * crit(logits.permute(0, 2, 1), ids[:, 1:].to(m.device))
        losses.append((tok_loss.sum(-1) / m.sum(-1)).cpu())
        nbytes = torch.tensor([len(t.encode("utf-8")) for t in texts[i:i + bs]], dtype=torch.float32)
        bpbs.append((tok_loss.sum(-1).cpu() * math.log2(math.e)) / nbytes)
        pooled = []
        for h in out.hidden_states:
            w = am.to(h.device).unsqueeze(-1).to(h.dtype)
            pooled.append(((h * w).sum(1) / w.sum(1)).float().cpu())                   # per layer: B x D
        feats.append(torch.stack(pooled, 1).to(torch.bfloat16))                         # B x L x D
        del out, logits
    res = dict(feats=torch.cat(feats), num_params=int(n_params),
               loss=torch.cat(losses).mean().item(), bpb=torch.cat(bpbs).mean().item(),
               precision="bf16", placement=plan["mode"], weights_gb=plan["weights_gb"], batch_size=bs, seq_len=T)
    del model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(offload_dir, ignore_errors=True)
    return res

@torch.no_grad()
def extract_vision(name, images, batch_size=64, pretrained=True):
    import timm
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform
    model = timm.create_model(name, pretrained=pretrained).eval()
    dev = "cuda" if torch.cuda.is_available() else "cpu"; model = model.to(dev)
    n_params = sum(p.numel() for p in model.parameters())
    tf = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
    store = {}
    hooks = [blk.register_forward_hook(lambda m, i, o, k=k: store.__setitem__(k, o[:, 0, :].float().cpu()))
             for k, blk in enumerate(model.blocks)]
    feats = []
    for i in range(0, len(images), batch_size):
        x = torch.stack([tf(im.convert("RGB")) for im in images[i:i + batch_size]]).to(dev)
        store.clear(); model(x)
        feats.append(torch.stack([store[k] for k in range(len(model.blocks))], 1))
    for h in hooks: h.remove()
    res = dict(feats=torch.cat(feats).to(torch.bfloat16), num_params=int(n_params))
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

def clear_hf_cache(name=None):
    """Delete downloaded LLM weights: the model's own folder (or all of them if name is None), any copy in the
    Hugging Face hub cache (located via huggingface_hub itself, wherever HF_HOME points), and the xet chunk cache."""
    from huggingface_hub import constants
    hub = constants.HF_HUB_CACHE
    if name is None:
        shutil.rmtree(MODEL_DIR, ignore_errors=True)
        for d in glob.glob(os.path.join(hub, "models--*")):
            shutil.rmtree(d, ignore_errors=True)
    else:
        shutil.rmtree(local_model_dir(name), ignore_errors=True)
        shutil.rmtree(os.path.join(hub, "models--" + name.replace("/", "--")), ignore_errors=True)
    shutil.rmtree(getattr(constants, "HF_XET_CACHE", os.path.join(constants.HF_HOME, "xet")), ignore_errors=True)

In [ ]:
# ============================== DATA (WIT-1024, identical to E2) ==============================
from datasets import load_dataset
ds = load_dataset("minhuh/prh", revision="wit_1024", split="train")
texts = [str(x["text"][0]) for x in ds]
N = len(texts); assert N == 1024
log(f"dataset: {N} captions; first: {texts[0][:80]!r}")
def fpath(model, readout):
    return os.path.join(FEAT_DIR, f"{model.replace('/', '_')}__{readout}.pt")

In [ ]:
# ============================== EXTRACT (one Python process per LLM, as in v4) ==============================
WORK_DIR = "/content/prh_work"; os.makedirs(WORK_DIR, exist_ok=True)
open(os.path.join(WORK_DIR, "prh_extract_v4.py"), "w").write('# ===================== Feature extraction (Huh et al. protocol), exact bf16 =====================\n# Language: every hidden state (embeddings + each block), mean-pooled over non-pad tokens,\n#           left padding to the longest caption in the dataset, bf16 weights. NO quantization:\n#           models larger than the GPU run with accelerate CPU/disk offload, which streams the bf16\n#           weights to the GPU layer by layer, so the arithmetic is identical to an all-GPU run.\n# Vision:   CLS token after every transformer block (= Huh\'s "blocks.{i}.add_1" node), timm transforms.\nimport gc, glob, math, os, shutil, time\nimport torch, transformers\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\ndef _dtype_kw(dtype):\n    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])\n    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}\n\ndef load_tokenizer(name, path=None):\n    path = path or name\n    try:\n        tok = AutoTokenizer.from_pretrained(path)\n    except Exception as e:                      # e.g. OpenLLaMA fast-tokenizer conversion issues\n        print("  fast tokenizer failed, using slow:", repr(e)[:120])\n        tok = AutoTokenizer.from_pretrained(path, use_fast=False)\n    if "huggyllama" in name:\n        tok.pad_token = tok.unk_token           # Huh et al. set "[PAD]" (not in vocab); padded positions are masked either way\n    elif tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "left"\n    return tok\n\ndef weights_gb(name):\n    """Exact checkpoint size from the Hub metadata (safetensors if present, else .bin)."""\n    from huggingface_hub import HfApi\n    info = HfApi().model_info(name, files_metadata=True)\n    files = [(s.rfilename, s.size or 0) for s in info.siblings]\n    st = [sz for f, sz in files if f.endswith(".safetensors")]\n    bn = [sz for f, sz in files if f.endswith(".bin") and "training_args" not in f]\n    return (sum(st) if st else sum(bn)) / 1e9, ("safetensors" if st else "bin")\n\ndef plan_placement(name, gpu_headroom_gb=14.0, cpu_headroom_gb=16.0, disk_headroom_gb=8.0):\n    """Decide GPU / GPU+CPU / GPU+CPU+disk placement for exact bf16 loading, or a reason to skip.\n    Memory budgets are computed in GiB (2**30 bytes), the unit accelerate\'s max_memory uses; disk in GB.\n    14 GiB GPU headroom = 7 GiB activations (extract_llm sizes the batch to fit) + CUDA contexts, the layer\n    being streamed in from CPU during offload, and allocator fragmentation. Offload is bit-exact\n    (verified on the GPU by the smoke test), so erring towards offload only costs time."""\n    import psutil\n    GiB = 2 ** 30\n    w, fmt = weights_gb(name)                                   # GB (1e9), from the Hub metadata\n    w_gib = w * 1e9 / GiB\n    gpu = (torch.cuda.get_device_properties(0).total_memory / GiB) if torch.cuda.is_available() else 0.0\n    cpu = psutil.virtual_memory().available / GiB\n    disk = shutil.disk_usage("/").free / 1e9\n    plan = dict(weights_gb=round(w, 1), weights_gib=round(w_gib, 1), format=fmt, gpu_gib=round(gpu, 1),\n                cpu_free_gib=round(cpu, 1), disk_free_gb=round(disk, 1))\n    if disk < w + disk_headroom_gb:\n        return dict(plan, mode="skip", reason=f"download needs {w + disk_headroom_gb:.0f} GB disk, {disk:.0f} GB free")\n    gpu_budget, cpu_budget = max(gpu - gpu_headroom_gb, 0), max(cpu - cpu_headroom_gb, 0)\n    if w_gib <= gpu_budget:\n        return dict(plan, mode="gpu", max_memory=None)\n    mm = {0: f"{int(gpu_budget)}GiB", "cpu": f"{int(cpu_budget)}GiB"}\n    if w_gib <= int(gpu_budget) + int(cpu_budget):\n        return dict(plan, mode="gpu+cpu", max_memory=mm)\n    spill_gb = (w_gib - int(gpu_budget) - int(cpu_budget)) * GiB / 1e9\n    if disk - w - disk_headroom_gb > spill_gb * 1.1:\n        return dict(plan, mode="gpu+cpu+disk", max_memory=mm)\n    return dict(plan, mode="skip", reason=f"needs {spill_gb:.0f} GB disk offload beyond the download; not enough disk")\n\nMODEL_DIR = "/content/hf_models"\n\ndef local_model_dir(name):\n    return os.path.join(MODEL_DIR, name.replace("/", "--"))\n\ndef download_llm(name):\n    """Download exactly one copy of the weights (the format weights_gb counted) into a dedicated folder.\n    Skips duplicate formats (Meta\'s original/*.pth, .msgpack, .h5), which would double the disk needed."""\n    from huggingface_hub import snapshot_download\n    _, fmt = weights_gb(name)\n    weights = ["*.safetensors"] if fmt == "safetensors" else ["*.bin"]\n    return snapshot_download(name, local_dir=local_model_dir(name),\n                             allow_patterns=["*.json", "*.model", "*.tiktoken", "*.txt"] + weights,\n                             ignore_patterns=["original/*", "*.pth", "*.pt", "*.msgpack", "*.h5", "*.onnx",\n                                              "training_args.bin"])\n\ndef load_llm(path, plan, offload_dir):\n    kw = dict(device_map="auto", **_dtype_kw(torch.bfloat16))\n    if plan["mode"] != "gpu":\n        # JSON turns the GPU key 0 into "0"; accelerate needs the integer device index\n        kw["max_memory"] = {(int(k) if str(k).isdigit() else k): v for k, v in plan["max_memory"].items()}\n    if plan["mode"] == "gpu+cpu+disk":\n        os.makedirs(offload_dir, exist_ok=True); kw["offload_folder"] = offload_dir\n    return AutoModelForCausalLM.from_pretrained(path, **kw).eval()\n\n@torch.no_grad()\ndef extract_llm(name, texts, plan, offload_dir="/content/offload", batch_size=32, act_budget_gb=7.0):\n    path = download_llm(name)                   # one explicit folder per model, deleted by the notebook afterwards\n    tok = load_tokenizer(name, path)\n    model = load_llm(path, plan, offload_dir)\n    n_params = sum(p.numel() for p in model.parameters())\n    tokens = tok(texts, padding="longest", return_tensors="pt")\n    T = tokens["input_ids"].shape[1]; V = model.get_output_embeddings().weight.shape[0]\n    # bf16 logits + their fp32 copy for the loss: 6 bytes per (token, vocab) entry; keep under the budget\n    # Batch size from the activation memory of THIS model, so it fits in the GPU headroom left by the weights.\n    # Per token: hidden states of every layer (kept until the forward ends because output_hidden_states=True),\n    # logits in bf16 + their fp32 copy for the loss, MLP intermediates, residual/attention scratch; per sequence:\n    # attention scores (fp32 upper bound). The batch size never changes the per-caption math (no batch statistics).\n    c = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config\n    L, D = c.num_hidden_layers, c.hidden_size\n    I = getattr(c, "intermediate_size", None) or 4 * D\n    H = getattr(c, "num_attention_heads", 1)\n    per_seq = T * ((L + 1) * D * 2 + V * 6 + 4 * I * 2 + 8 * D * 2) + H * T * T * 4\n    bs = max(1, min(batch_size, int(act_budget_gb * 1e9 // per_seq)))\n    print(f"  activation estimate {per_seq / 1e6:.0f} MB per caption -> batch size {bs}", flush=True)\n    dev = "cuda:0" if torch.cuda.is_available() else "cpu"   # accelerate hooks move inputs to each module\'s device\n    feats, losses, bpbs = [], [], []\n    crit = torch.nn.CrossEntropyLoss(reduction="none")\n    for i in range(0, len(texts), bs):\n        ids = tokens["input_ids"][i:i + bs].to(dev).long()\n        am = tokens["attention_mask"][i:i + bs].to(dev).long()\n        out = model(input_ids=ids, attention_mask=am, output_hidden_states=True)\n        m = am[:, :-1].float()\n        logits = out.logits[:, :-1].float().to(m.device)\n        tok_loss = m * crit(logits.permute(0, 2, 1), ids[:, 1:].to(m.device))\n        losses.append((tok_loss.sum(-1) / m.sum(-1)).cpu())\n        nbytes = torch.tensor([len(t.encode("utf-8")) for t in texts[i:i + bs]], dtype=torch.float32)\n        bpbs.append((tok_loss.sum(-1).cpu() * math.log2(math.e)) / nbytes)\n        pooled = []\n        for h in out.hidden_states:\n            w = am.to(h.device).unsqueeze(-1).to(h.dtype)\n            pooled.append(((h * w).sum(1) / w.sum(1)).float().cpu())                   # per layer: B x D\n        feats.append(torch.stack(pooled, 1).to(torch.bfloat16))                         # B x L x D\n        del out, logits\n    res = dict(feats=torch.cat(feats), num_params=int(n_params),\n               loss=torch.cat(losses).mean().item(), bpb=torch.cat(bpbs).mean().item(),\n               precision="bf16", placement=plan["mode"], weights_gb=plan["weights_gb"], batch_size=bs, seq_len=T)\n    del model; gc.collect(); torch.cuda.empty_cache()\n    shutil.rmtree(offload_dir, ignore_errors=True)\n    return res\n\n@torch.no_grad()\ndef extract_vision(name, images, batch_size=64, pretrained=True):\n    import timm\n    from timm.data import resolve_data_config\n    from timm.data.transforms_factory import create_transform\n    model = timm.create_model(name, pretrained=pretrained).eval()\n    dev = "cuda" if torch.cuda.is_available() else "cpu"; model = model.to(dev)\n    n_params = sum(p.numel() for p in model.parameters())\n    tf = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))\n    store = {}\n    hooks = [blk.register_forward_hook(lambda m, i, o, k=k: store.__setitem__(k, o[:, 0, :].float().cpu()))\n             for k, blk in enumerate(model.blocks)]\n    feats = []\n    for i in range(0, len(images), batch_size):\n        x = torch.stack([tf(im.convert("RGB")) for im in images[i:i + batch_size]]).to(dev)\n        store.clear(); model(x)\n        feats.append(torch.stack([store[k] for k in range(len(model.blocks))], 1))\n    for h in hooks: h.remove()\n    res = dict(feats=torch.cat(feats).to(torch.bfloat16), num_params=int(n_params))\n    del model; gc.collect(); torch.cuda.empty_cache()\n    return res\n\ndef clear_hf_cache(name=None):\n    """Delete downloaded LLM weights: the model\'s own folder (or all of them if name is None), any copy in the\n    Hugging Face hub cache (located via huggingface_hub itself, wherever HF_HOME points), and the xet chunk cache."""\n    from huggingface_hub import constants\n    hub = constants.HF_HUB_CACHE\n    if name is None:\n        shutil.rmtree(MODEL_DIR, ignore_errors=True)\n        for d in glob.glob(os.path.join(hub, "models--*")):\n            shutil.rmtree(d, ignore_errors=True)\n    else:\n        shutil.rmtree(local_model_dir(name), ignore_errors=True)\n        shutil.rmtree(os.path.join(hub, "models--" + name.replace("/", "--")), ignore_errors=True)\n    shutil.rmtree(getattr(constants, "HF_XET_CACHE", os.path.join(constants.HF_HOME, "xet")), ignore_errors=True)\n')
open(os.path.join(WORK_DIR, "e2p_extract.py"), "w").write('# ===================== E2-prompt: prompted last-token readouts for decoder LLMs (exact bf16) =====================\n# Each caption is wrapped in a template; the readout is the hidden state of the LAST token at every layer\n# (embeddings + each block; the last entry is after the final norm). Left padding, so position -1 is always real.\n# Weights: exactly as in the E2 audit (bf16, GPU or GPU+CPU offload via prh_extract_v4.plan_placement / load_llm).\nimport gc, os, torch\nfrom prh_extract_v4 import load_tokenizer, download_llm, load_llm\n\nPROMPTS = {\n    # main readout: PromptEOL (Jiang et al. 2023)\n    "eol": \'This sentence : "{}" means in one word:"\',\n    # second summarising prompt, different wording\n    "eol_alt": \'This image caption : "{}" can be summarized in one word:"\',\n    # control: same frame and length, but asks for a continuation, not a summary\n    "neutral": \'This sentence : "{}" was written down and then\',\n    # control: caption repeated, so the last token has read the whole caption once already (no instruction)\n    "repeat": \'{}\\n{}\',\n}\n\ndef fill(template, caption):\n    return template.replace("{}", caption)\n\n@torch.no_grad()\ndef extract_prompted(name, texts, plan, readouts, offload_dir="/content/offload", batch_size=32, act_budget_gb=7.0):\n    path = download_llm(name)\n    tok = load_tokenizer(name, path); tok.padding_side = "left"\n    model = load_llm(path, plan, offload_dir)\n    base = model.base_model                              # transformer without the LM head: no logits needed\n    c = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config\n    L, D = c.num_hidden_layers, c.hidden_size\n    I = getattr(c, "intermediate_size", None) or 4 * D\n    H = getattr(c, "num_attention_heads", 1)\n    n_params = sum(p.numel() for p in model.parameters())\n    dev = "cuda:0" if torch.cuda.is_available() else "cpu"\n    # some tokenizers append an EOS token; the readout must be the template\'s own final token, so step back over it\n    probe = tok("a b")["input_ids"]\n    pos = -2 if (tok.eos_token_id is not None and len(probe) > 1 and probe[-1] == tok.eos_token_id) else -1\n    out = {}\n    for r in readouts:\n        inp = [fill(PROMPTS[r], t) for t in texts]\n        T = max(len(x) for x in tok(inp)["input_ids"])\n        per_seq = T * ((L + 1) * D * 2 + 4 * I * 2 + 8 * D * 2) + H * T * T * 4\n        # Offloaded models (65B/70B) also stream layer weights through the GPU; use half the activation budget there.\n        budget = act_budget_gb if plan["mode"] == "gpu" else act_budget_gb / 2\n        bs = max(1, min(batch_size, int(budget * 1e9 // per_seq)))\n        print(f"  [{r}] {T} tokens max, ~{per_seq / 1e6:.0f} MB per caption -> batch size {bs}", flush=True)\n        feats, last_ids = [], []\n        for i in range(0, len(inp), bs):\n            b = tok(inp[i:i + bs], padding="longest", return_tensors="pt")\n            ids, am = b["input_ids"].to(dev), b["attention_mask"].to(dev)\n            assert bool(am[:, -1].all()), "left padding expected"\n            hs = base(input_ids=ids, attention_mask=am, output_hidden_states=True).hidden_states\n            feats.append(torch.stack([h[:, pos].float().cpu() for h in hs], 1).to(torch.bfloat16))\n            last_ids.append(ids[:, pos].cpu())\n            del hs\n        last_ids = torch.cat(last_ids)\n        out[r] = dict(feats=torch.cat(feats), readout=r, template=PROMPTS[r], num_params=int(n_params),\n                      precision="bf16", placement=plan["mode"], batch_size=bs, max_tokens=T,\n                      last_token=tok.decode(last_ids[:1]), last_token_unique=int(last_ids.unique().numel()), readout_position=pos)\n    del model, base; gc.collect(); torch.cuda.empty_cache()\n    return out\n')
WORKER = os.path.join(WORK_DIR, "e2p_worker.py")
open(WORKER, "w").write("""import json, os, sys, torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from e2p_extract import extract_prompted
name, feat_dir, plan_fp, texts_fp, readouts = sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4], sys.argv[5].split(",")
out = extract_prompted(name, json.load(open(texts_fp)), json.load(open(plan_fp)), readouts)
for r, d in out.items():
    fp = os.path.join(feat_dir, name.replace("/", "_") + "__" + r + ".pt")
    torch.save(d, fp + ".tmp"); os.replace(fp + ".tmp", fp)
""")
TEXTS_FP = os.path.join(WORK_DIR, "texts.json"); json.dump(texts, open(TEXTS_FP, "w"))
def free_disk_gb(): return shutil.disk_usage("/").free / 1e9
def _size(m):
    try: return weights_gb(m)[0]
    except Exception: return 1e9
def wanted(m):
    return [r for r in READOUTS if r == "eol" or _size(m) <= CONTROLS_MAX_GB]
todo = [m for m in LLMS if any(not os.path.exists(fpath(m, r)) for r in wanted(m))]
if RUN_ONLY:
    only = [RUN_ONLY] if isinstance(RUN_ONLY, str) else list(RUN_ONLY); todo = [m for m in todo if m in only]
if RUN_EXCEPT:
    excl = [RUN_EXCEPT] if isinstance(RUN_EXCEPT, str) else list(RUN_EXCEPT); todo = [m for m in todo if m not in excl]
log("to extract:", todo)
if todo:
    d0 = free_disk_gb(); clear_hf_cache(None); log(f"cleared old downloads: free disk {d0:.0f} -> {free_disk_gb():.0f} GB")
for name in sorted(todo, key=_size):
    rs = [r for r in wanted(name) if not os.path.exists(fpath(name, r))]
    try:
        plan = plan_placement(name)
    except Exception as e:
        log("LLM FAILED (pre-flight):", name, repr(e)[:300]); continue
    log("pre-flight:", name, {k: v for k, v in plan.items() if k != "max_memory"}, "| readouts:", rs)
    if plan["mode"] == "skip":
        log(f"SKIPPING {name}: {plan['reason']}"); continue
    import gc; gc.collect(); torch.cuda.empty_cache()
    t0 = time.time(); plan_fp = os.path.join(WORK_DIR, "plan.json"); json.dump(plan, open(plan_fp, "w"))
    wlog = os.path.join(RUN, "worker_" + name.replace("/", "_") + ".txt")
    with open(wlog, "w") as fh:
        rc = subprocess.run([sys.executable, WORKER, name, FEAT_DIR, plan_fp, TEXTS_FP, ",".join(rs)], stdout=fh, stderr=subprocess.STDOUT,
                            env=dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")).returncode
    clear_hf_cache(name); shutil.rmtree("/content/offload", ignore_errors=True)
    if rc == 0 and all(os.path.exists(fpath(name, r)) for r in rs):
        d = torch.load(fpath(name, rs[0]), mmap=True)
        log(f"llm done: {name} readouts={rs} feats={tuple(d['feats'].shape)} placement={d['placement']} batch={d['batch_size']} "
            f"({time.time()-t0:.0f}s) | free disk {free_disk_gb():.0f} GB")
    else:
        tail = open(wlog, errors="ignore").read().replace("\r", "\n").strip().splitlines()[-12:]
        log(f"LLM FAILED: {name} (exit code {rc}):\n    " + "\n    ".join(tail))

In [ ]:
# ============================== ALIGN (E2 engine, zero-variance layers dropped) ==============================
g = torch.Generator().manual_seed(SEED)
PERMS = [torch.randperm(N, generator=g).to(DEV) for _ in range(N_PERMS)]
_split = torch.randperm(N, generator=g)
HALF = (_split[: N // 2].to(DEV), _split[N // 2:].to(DEV))
PERMS_HALF = [torch.randperm(N - N // 2, generator=g).to(DEV) for _ in range(N_PERMS)]
DROPPED = {}
def stats(path, variant, tag):
    f = torch.load(path)["feats"].to(DEV).float()
    layers = preprocess(f, variant)
    keep = [l for l, X in enumerate(layers) if float((X - X.mean(0)).pow(2).sum()) > 1e-6 * X.shape[0]]
    if len(keep) < len(layers): DROPPED[f"{tag}|{variant}"] = [l for l in range(len(layers)) if l not in keep]
    s = SideStats([layers[l] for l in keep], k=TOPK, half_idx=HALF); del f, layers; return s

for variant in VARIANTS_P:
    vs = None
    for r in READOUTS:
        out_csv = os.path.join(RES_DIR, f"align_{r}_{variant}.csv")
        done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["llm"])
        todo_a = [m for m in LLMS if os.path.exists(fpath(m, r)) and m not in set(done.llm)]
        if not todo_a: continue
        if vs is None:
            vs = {v: stats(os.path.join(E2_FEAT, v.replace("/", "_") + ".pt"), variant, v) for v in VISION}
        for m in todo_a:
            t1 = time.time(); xs = stats(fpath(m, r), variant, f"{m}/{r}"); rows = []
            for v, ys in vs.items():
                o, _ = align_pair(xs, ys, PERMS, alpha=ALPHA, perms_half=PERMS_HALF)
                rows.append(dict(variant=variant, readout=r, llm=m, vision=v, **{a: (str(b) if isinstance(b, tuple) else b) for a, b in o.items()}))
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(out_csv, index=False)
            log(f"[{variant}/{r}] {m}: {len(rows)} pairs ({time.time()-t1:.0f}s)")
            del xs; torch.cuda.empty_cache()
    del vs; torch.cuda.empty_cache()
log("zero-variance layers dropped:", DROPPED)

In [ ]:
# ============================== CHECKER ==============================
checks = []
def check(name, ok, detail=""):
    checks.append(dict(check=name, result="PASS" if ok else "FAIL", detail=str(detail)[:200]))
wl = open(LOG).read()
check("no extraction failures", "FAILED" not in wl, [l for l in wl.splitlines() if "FAILED" in l][:3])
exp = [(m, r) for m in LLMS for r in wanted(m)]
missing = [f"{m}/{r}" for m, r in exp if not os.path.exists(fpath(m, r))]
check("feature files present (this session's models)", not missing, missing[:6])
bad, lt = [], {}
for m, r in exp:
    if os.path.exists(fpath(m, r)):
        d = torch.load(fpath(m, r), mmap=True)
        if d["feats"].shape[0] != N or not torch.isfinite(d["feats"].float()).all(): bad.append(f"{m}/{r}")
        lt[f"{m.split('/')[-1]}/{r}"] = (d["last_token"], d["last_token_unique"])
check("features: 1024 rows, all finite", not bad, bad[:5])
fixed_end = {k: v for k, v in lt.items() if not k.endswith("/repeat")}
expected_end = {"eol": '"', "eol_alt": '"', "neutral": "then"}
ok_end = all(u == 1 and v[0].strip().endswith(expected_end[k.split("/")[-1]]) for k, v in fixed_end.items() for u in [v[1]])
check("prompted readouts sit on the template's final token for every caption", ok_end, {k: v for k, v in list(fixed_end.items())[:6]})
for m in LLMS:
    rs = [r for r in wanted(m) if os.path.exists(fpath(m, r))]
    if len(rs) >= 2:
        a, b = (torch.load(fpath(m, r))["feats"][:, -1].float() for r in rs[:2])
        check(f"readouts differ ({m.split('/')[-1]}: {rs[0]} vs {rs[1]})", not torch.allclose(a, b), float((a - b).abs().max()))
res_p = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(RES_DIR, "align_*.csv"))], ignore_index=True)
check("calibrated CKA in [0, 1] and <= raw, no NaN", bool(((res_p.cka_cal >= 0) & (res_p.cka_cal <= res_p.cka_max + 1e-9)).all()) and not res_p.cka_max.isna().any())
tp = np.sqrt(res_p.nullmax_reff_x * res_p.nullmax_reff_y) / N
check("threshold = chance level of the highest-rank layer pair (r > 0.99)", float(np.corrcoef(tp, res_p.cka_tau)[0, 1]) > 0.99,
      round(float(np.corrcoef(tp, res_p.cka_tau)[0, 1]), 5))
m0 = next(m for m in LLMS if os.path.exists(fpath(m, "eol"))); v0 = VISION[0]
xs = stats(fpath(m0, "eol"), "huh", "null-check"); zero = 0
fv = torch.load(os.path.join(E2_FEAT, v0.replace("/", "_") + ".pt"))["feats"].to(DEV).float()
for s_ in range(10):
    p = torch.randperm(N, generator=torch.Generator().manual_seed(100 + s_)).to(DEV)
    ys = SideStats(preprocess(fv[p], "huh"), k=TOPK, half_idx=HALF)
    o, _ = align_pair(xs, ys, PERMS, alpha=ALPHA); zero += int(o["cka_cal"] == 0)
check("null: shuffled pairs give calibrated CKA = 0 (>= 9/10)", zero >= 9, f"{zero}/10")
e2 = pd.read_csv(os.path.join(E2_RES, "alignment_all.csv"))
check("E2 mean-pooled baseline available for these LLMs", set(LLMS) <= set(e2.llm), sorted(set(LLMS) - set(e2.llm)))
checks = pd.DataFrame(checks); checks.to_csv(os.path.join(RUN, "checker.csv"), index=False)
print(checks.to_string(index=False)); log("checker:", (checks.result == "PASS").sum(), "PASS /", len(checks))
if SMOKE_TEST:
    import zipfile
    zp = os.path.join(RUN, "e2p_smoke_results.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in [LOG, os.path.join(RUN, "checker.csv")] + glob.glob(os.path.join(RES_DIR, "*.csv")) + glob.glob(os.path.join(RUN, "worker_*.txt")):
            z.write(f, os.path.relpath(f, RUN))
    if IN_COLAB:
        from google.colab import files; files.download(zp)
    raise SystemExit("Smoke test finished: send e2p_smoke_results.zip, then set SMOKE_TEST = False.")

In [ ]:
# ============================== PRE-REGISTERED ANALYSIS (two-way bootstrap: LLMs x ViTs) ==============================
import matplotlib; import matplotlib.pyplot as plt
meta = pd.read_csv(os.path.join(E2_RES, "llm_meta.csv"))[["llm", "bpb", "llm_params"]]
base_cols = ["variant", "llm", "vision", "cka_max", "cka_cal", "cka_tau", "ucka_heldout", "mknn_max"]
mean_rows = e2[(e2.variant.isin(VARIANTS_P)) & (e2.precision == "bf16")][base_cols].assign(readout="mean")
allr = pd.concat([mean_rows, res_p[base_cols + ["readout"]]], ignore_index=True).merge(meta, on="llm")
allr["family"] = allr.llm.map(llm_family); allr["log2p"] = np.log2(allr.llm_params.astype(float)); allr["cap"] = -allr.bpb
allr.to_csv(os.path.join(RES_DIR, "e2p_all.csv"), index=False)
MET = ["cka_cal", "ucka_heldout", "mknn_max", "cka_max", "cka_tau"]
rng = np.random.default_rng(SEED)

def mat(variant, readout, y, llms, vits):
    d = allr[(allr.variant == variant) & (allr.readout == readout)].pivot_table(index="llm", columns="vision", values=y)
    return d.reindex(index=llms, columns=vits).values
lmeta = allr.groupby("llm").agg(log2p=("log2p", "first"), cap=("cap", "first"), family=("family", "first"))

def slope_fe_arr(Y, Li, Vj, logp, fam):
    """slope of Y on log2(params) with ViT (column) and family fixed effects; rows Li, columns Vj (copies = own levels)."""
    nL, nV = len(Li), len(Vj)
    y = Y[np.ix_(Li, Vj)].ravel()
    x = np.repeat(logp[Li], nV)
    vfe = np.tile(np.eye(nV), (nL, 1))
    fl = fam[Li]; levels = np.unique(fl)[1:]
    ffe = np.repeat(np.stack([(fl == lv).astype(float) for lv in levels], 1), nV, 0) if len(levels) else np.zeros((nL * nV, 0))
    X = np.column_stack([x, vfe, ffe]); ok = ~np.isnan(y)
    return float(np.linalg.lstsq(X[ok], y[ok], rcond=None)[0][0])
def cap_r_arr(Y, Li, Vj, cap):
    m = np.nanmean(Y[np.ix_(Li, Vj)], 1); return float(np.corrcoef(m, cap[Li])[0, 1])

def boot_pair(stat, nL, nV, B=B_BOOT):
    out = []
    for _ in range(B):
        Li, Vj = rng.integers(0, nL, nL), rng.integers(0, nV, nV)
        try: out.append(stat(Li, Vj))
        except Exception: out.append(np.nan)
    return np.array(out)
def ci(a): return float(np.nanpercentile(a, 2.5)), float(np.nanpercentile(a, 97.5))

rows = []
vits = sorted(allr.vision.unique())
for variant in VARIANTS_P:
    # R1 / R2: eol vs mean on the LLMs that have the eol readout
    llms = sorted(set(allr[(allr.readout == "eol") & (allr.variant == variant)].llm) & set(allr[allr.readout == "mean"].llm))
    if len(llms) >= 3:
        logp, cap = lmeta.loc[llms, "log2p"].values, lmeta.loc[llms, "cap"].values
        fam = lmeta.loc[llms, "family"].values; full = (np.arange(len(llms)), np.arange(len(vits)))
        for y in MET:
            Ye, Ym = mat(variant, "eol", y, llms, vits), mat(variant, "mean", y, llms, vits)
            s_e, s_m = slope_fe_arr(Ye, *full, logp, fam), slope_fe_arr(Ym, *full, logp, fam)
            c_e, c_m = cap_r_arr(Ye, *full, cap), cap_r_arr(Ym, *full, cap)
            bs_s = boot_pair(lambda Li, Vj: slope_fe_arr(Ye, Li, Vj, logp, fam) - slope_fe_arr(Ym, Li, Vj, logp, fam), len(llms), len(vits))
            bs_c = boot_pair(lambda Li, Vj: cap_r_arr(Ye, Li, Vj, cap) - cap_r_arr(Ym, Li, Vj, cap), len(llms), len(vits))
            rows.append(dict(test=f"R1 slope per doubling (family + ViT FE), {len(llms)} LLMs", variant=variant, metric=y, a="eol", b="mean",
                             value_a=s_e, value_b=s_m, delta=s_e - s_m, ci_low=ci(bs_s)[0], ci_high=ci(bs_s)[1],
                             primary=(variant == "huh" and y == "cka_cal")))
            rows.append(dict(test=f"R2 Pearson with capability (LLM means), {len(llms)} LLMs", variant=variant, metric=y, a="eol", b="mean",
                             value_a=c_e, value_b=c_m, delta=c_e - c_m, ci_low=ci(bs_c)[0], ci_high=ci(bs_c)[1],
                             primary=(variant == "huh" and y == "cka_cal")))
    # R3: controls (mean over all LLM x ViT pairs where both readouts exist)
    for a, b in (("eol", "neutral"), ("eol", "repeat"), ("eol_alt", "eol"), ("eol", "mean"), ("neutral", "mean"), ("repeat", "mean")):
        both = sorted(set(allr[(allr.readout == a) & (allr.variant == variant)].llm) & set(allr[(allr.readout == b) & (allr.variant == variant)].llm))
        if len(both) < 2: continue
        for y in ["cka_cal", "ucka_heldout", "mknn_max"]:
            Ya, Yb = mat(variant, a, y, both, vits), mat(variant, b, y, both, vits)
            dm = lambda Li, Vj: float(np.nanmean(Ya[np.ix_(Li, Vj)]) - np.nanmean(Yb[np.ix_(Li, Vj)]))
            d0 = dm(np.arange(len(both)), np.arange(len(vits))); bs = boot_pair(dm, len(both), len(vits))
            rows.append(dict(test=f"R3 {a} - {b} (mean over pairs), {len(both)} LLMs", variant=variant, metric=y, a=a, b=b,
                             value_a=float(np.nanmean(Ya)), value_b=float(np.nanmean(Yb)), delta=d0, ci_low=ci(bs)[0], ci_high=ci(bs)[1],
                             primary=(variant == "huh" and y == "cka_cal" and (a, b) == ("eol", "neutral"))))
tests = pd.DataFrame(rows); tests.to_csv(os.path.join(RES_DIR, "e2p_tests.csv"), index=False)
print(tests.round(4).to_string(index=False))

# R4: per-LLM means (huh), mean vs eol, for every family
per = allr[allr.variant == "huh"].groupby(["family", "llm", "llm_params", "readout"])[MET].mean().reset_index()
per.to_csv(os.path.join(RES_DIR, "e2p_per_llm.csv"), index=False)
fams = ["BLOOMZ", "OpenLLaMA", "LLaMA", "Llama-3", "Mistral", "OLMo", "Gemma"]
cols = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6"]
fig, axs = plt.subplots(1, 4, figsize=(15, 3.6), sharey=False)
for ax, (rd, y, ttl) in zip(axs, [("mean", "cka_cal", "mean pooling: calibrated CKA"), ("eol", "cka_cal", "PromptEOL: calibrated CKA"),
                                  ("mean", "cka_tau", "mean pooling: threshold τ"), ("eol", "cka_tau", "PromptEOL: threshold τ")]):
    for f_, c_ in zip(fams, cols):
        gg = per[(per.family == f_) & (per.readout == rd)].sort_values("llm_params")
        if len(gg): ax.plot(gg.llm_params / 1e9, gg[y], "o-", color=c_, label=f_)
    ax.set_xscale("log"); ax.set_title(ttl); ax.set_xlabel("LLM parameters (B)"); ax.grid(alpha=.3)
axs[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_e2p_families.png"), dpi=150); plt.show()

In [ ]:
# ============================== PACKAGE ==============================
import zipfile
zp = os.path.join(RUN, "e2p_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [LOG, os.path.join(RUN, "checker.csv")] + glob.glob(os.path.join(RES_DIR, "*.csv")) + glob.glob(os.path.join(FIG_DIR, "*.png")):
        if os.path.exists(f): z.write(f, os.path.relpath(f, RUN))
log("packaged", zp, f"{os.path.getsize(zp) / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files; files.download(zp)